# Lesson 19 — Practice: HTTP with requests

Work in `lessons/lesson-19-requests/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. The server is the same `local_api.py` as L18 (provided, do not
modify); only your client library changes. Install once: `pip install requests`.


## E1 — Build query URLs with params=

## Intent
**Inspect what will be sent: build an encoded query URL from a params dict, never hand-writing any encoding.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
A query dict must become a URL-safe string (`coffee latte` → `coffee+latte`).
Write `build_url(base, path, params)` that prepares a GET request and returns
the final URL as a string.

**File tree**

```
url_parts.py          (you create)
```

**Interface contract**
`build_url(base: str, path: str, params: dict) -> str` — returns the prepared
`.url` for `GET base + path` with the given params.

**Constraints**
- **Build the URL with `requests.Request(...).prepare().url`; never format `?`/`&`/`+` by hand.**
- An empty params dict returns the bare URL with no trailing `?`.
- Driver imports `import url_parts as up`.

**Coverage**
`requests.Request` + `params=` + `.prepare().url` → lesson.md Core Concept 2.

**Expected** (Driver cell):
```
http://127.0.0.1:8000/search?q=coffee+latte&max=5
http://127.0.0.1:8000/weather?city=new+york
https://api.example.com/v1/items
```


In [2]:
import url_parts as up

cases = [
    ("http://127.0.0.1:8000", "/search", {"q": "coffee latte", "max": 5}),
    ("http://127.0.0.1:8000", "/weather", {"city": "new york"}),
    ("https://api.example.com", "/v1/items", {}),
]
for base, path, params in cases:
    print(up.build_url(base, path, params))

http://127.0.0.1:8000/search?q=coffee+latte&max=5
http://127.0.0.1:8000/weather?city=new+york
https://api.example.com/v1/items


## E2 — Read the status code and its family

## Intent
**Decide from the verdict: read `status_code` (which never raises by itself) and map it to its family with `match`.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
Every response carries a status code. Write `get_status` and `status_family`
so a caller can ask "what did the server decide, and which family is it in".

**File tree**

```
client_basics.py      (you create)
```

**Interface contract**
`get_status(base: str, path: str) -> int` — returns `resp.status_code`.
`status_family(code: int) -> str` — maps to `"2xx"` / `"3xx"` / `"4xx"` /
`"5xx"`, anything else → `"?"`.

**Constraints**
- **Use `requests.get(..., timeout=5)`; read `.status_code` directly — do NOT call `raise_for_status()` here.**
- `status_family` uses `match` on the first digit.
- Driver imports `import client_basics as cb` and starts its own `LocalAPI`.

**Coverage**
`requests.get` + `.status_code` → lesson.md Core Concept 3; `match` ranges → L3/L18 retrieval.

**Expected** (Driver cell):
```
/items -> 200 2xx
/items/999 -> 404 4xx
/echo -> 405 4xx
/nope -> 404 4xx
```


In [1]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    for path in ["/items", "/items/999", "/echo", "/nope"]:
        code = cb.get_status(api.base, path)
        print(path, "->", code, cb.status_family(code))
finally:
    api.stop()

/items -> 200 2xx
/items/999 -> 404 4xx
/echo -> 405 4xx
/nope -> 404 4xx


## E3 — Read a JSON body

## Intent
**Parse the evidence: return the response body as a dict via `.json()`, with a clean empty-dict fallback on failure.**

**Type:** REAL-WORLD — Easy — edits 1 file

**Task**
`/items` answers with a JSON object. Write `get_json(base, path)` that returns
the parsed body as a dict.

**File tree**

```
client_basics.py      (you edit)
```

**Interface contract**
`get_json(base: str, path: str) -> dict` — parsed body on success, `{}` when
the request fails at any point.

**Constraints**
- **Parse with `resp.json()` after `raise_for_status()`; catch `requests.RequestException` for the `{}` fallback.**
- Driver imports `import client_basics as cb`.

**Coverage**
`.json()` + `raise_for_status()` + `RequestException` parent → lesson.md Core Concepts 3/5/6.

**Expected** (Driver cell):
```
count = 4
first = coffee  last = latte
```


In [1]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    data = cb.get_json(api.base, "/items")
    print("count =", data["count"])
    print("first =", data["items"][0]["name"], " last =", data["items"][-1]["name"])
finally:
    api.stop()

count = 4
first = coffee  last = latte


## E4 — Inspect response headers

## Intent
**Read the meta before the body: report `Content-Type`, declared length, and actual body size.**

**Type:** REAL-WORLD — Easy — edits 1 file

**Task**
`resp.headers` is a case-insensitive header map. Write `get_headers` that
reports the content type, the declared length, and how many body bytes really
arrived.

**File tree**

```
client_basics.py      (you edit)
```

**Interface contract**
`get_headers(base: str, path: str) -> dict` — returns exactly
`{"content-type": ..., "content-length": ..., "body_bytes": ...}` where the
first two are header strings and `body_bytes` is `len(resp.content)`.

**Constraints**
- **Read headers with `resp.headers.get(...)`; measure the body with `len(resp.content)` — never `read()`.**
- Lowercase dict keys exactly as above.

**Coverage**
`.headers` / `.content` → lesson.md Core Concept 3.

**Expected** (Driver cell):
```
content-type: application/json; charset=utf-8
content-length header: 242
body bytes: 242
```


In [2]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    info = cb.get_headers(api.base, "/items")
    print("content-type:", info["content-type"])
    print("content-length header:", info["content-length"])
    print("body bytes:", info["body_bytes"])
finally:
    api.stop()

content-type: application/json; charset=utf-8
content-length header: 242
body bytes: 242


## E5 — Handle a timeout

## Intent
**Name the failure precisely: catch the timeout family and report which member fired.**

**Type:** REAL-WORLD — Easy — edits 1 file

**Task**
`/slow` sleeps ~0.5s. Write `probe_timeout` that asks with a tiny timeout and
reports what the request layer raises.

**File tree**

```
client_basics.py      (you edit)
```

**Interface contract**
`probe_timeout(base: str, path: str = "/slow", timeout: float = 0.05) -> str`
— returns `type(exc).__name__` of the caught timeout, or `"no-timeout"` if the
call somehow succeeds.

**Constraints**
- **Catch `requests.Timeout` (the parent); report the class name, never the raw message (ports vary).**
- `timeout=` must be passed to the request.

**Coverage**
`timeout=` + `requests.Timeout` family → lesson.md Core Concepts 5/6.

**Expected** (Driver cell):
```
ReadTimeout
```


In [1]:
import client_basics as cb
import local_api

api = local_api.LocalAPI()
try:
    print(cb.probe_timeout(api.base))
finally:
    api.stop()

ReadTimeout


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 64572)
Traceback (most recent call last):
  File "C:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 692, in process_request_thread
    self.finish_request(request, client_address)
  File "C:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\socketserver.py", line 761, in __init__
    self.handle()
  File "C:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\http\server.py", line 438, in handle
    self.handle_one_request()
  File "C:\Users\MSI\AppData\Local\Programs\Python\Python312\Lib\http\server.py", line 404, in handle_one_request
    self.raw_requestline = self.rfile.readline(65537)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MSI\AppData\Local

## M1 — Categorize errors with ApiError

## Intent
**Give every failure one typed shape: wrap the whole `requests` family into `ApiError` with status-or-None.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
Two very different failures look the same to a naive client: "the server
answered with an error" vs "nothing answered at all". Write one typed
`fetch_json` that separates them, plus a guard that rejects malformed paths
before any network call.

**File tree**

```
api_client.py         (you create)
```

**Interface contract**
`ApiError(message, status=None)` with `.message` / `.status` attributes.
`fetch_json(base: str, path: str) -> dict`:
- path that is not a `str` or does not start with `"/"` → `raise ApiError(f"bad path: {path!r}")` with `status=None`;
- HTTP error status → `raise ApiError(f"http {code}", status=code)`;
- unreachable host → `ApiError` with `status=None`;
- success → parsed dict.

**Constraints**
- **One `try` around get + `raise_for_status()` + `.json()`; handlers in order: `HTTPError`, then `ConnectionError`, then `Timeout`.**
- **The path guard runs before any network code and uses `or`, not `and`.**
- Driver imports `import api_client as ac`.

**Coverage**
`raise_for_status()` + `HTTPError/.response` + `ConnectionError` + `Timeout` → lesson.md Core Concepts 5/6; guard-operator idiom → WP2 retrieval.

**Expected** (Driver cell):
```
ok count = 4
http error: 404 http 404
bad path: bad path: ''
network error, status = None
```


In [4]:
import api_client as ac
import local_api

api = local_api.LocalAPI()
try:
    ok = ac.fetch_json(api.base, "/items")
    print("ok count =", ok["count"])
    try:
        ac.fetch_json(api.base, "/items/999")
    except ac.ApiError as e:
        print("http error:", e.status, e.message)
    try:
        ac.fetch_json(api.base, "")
    except ac.ApiError as e:
        print("bad path:", e.message)
    try:
        ac.fetch_json("http://127.0.0.1:1", "/items")
    except ac.ApiError as e:
        print("network error, status =", e.status)
finally:
    api.stop()

ok count = 4
http error: 404 http 404
bad path: bad path: ''
network error, status = None


## M2 — POST JSON to an API

## Intent
**Send structured data with `json=` and prove with a counter that POST is not idempotent.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
Up to now everything has been a GET. Write a small POST client that takes a
JSON payload, sends it as the request body, and returns the status plus the
parsed answer.

**File tree**

```
post_client.py        (you create)
```

**Interface contract**
`post_json(base: str, path: str, payload: dict) -> tuple[int, dict]` —
`(resp.status_code, resp.json())`.

**Constraints**
- **Send the payload with `json=payload` (which also sets the JSON content type); pass `timeout=`.**
- Three POSTs to `/count` must answer n = 1, 2, 3 — same request, different result.
- Driver imports `import post_client as pc` and shares one `LocalAPI` so the counter keeps counting.

**Coverage**
`requests.post` + `json=` + status/body return → lesson.md Core Concept 4.

**Expected** (Driver cell):
```
counts: [1, 2, 3]
201 {'received': {'note': 'hi'}}
```


In [1]:
import local_api
import post_client as pc

api = local_api.LocalAPI()
try:
    counts = [pc.post_json(api.base, "/count", {})[1]["n"] for _ in range(3)]
    print("counts:", counts)
    status, body = pc.post_json(api.base, "/echo", {"note": "hi"})
    print(status, body)
finally:
    api.stop()

counts: [1, 2, 3]
201 {'received': {'note': 'hi'}}


## M3 — Search with params= and export CSV

## Intent
**Combine network + files: fetch a filtered search with `params=` and persist the hits as CSV.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
Query strings have encoding rules you must never hand-apply. Write `search`
using `params=`, plus `save_csv` that writes the hits for a report.

**File tree**

```
search_client.py      (you create)
```

**Interface contract**
`search(base: str, q: str, max_items: int = 10) -> dict` — GETs
`/search` with `params={"q": q, "max": max_items}`, raises on bad status,
returns the parsed dict.
`save_csv(items: list[dict], path: Path) -> int` — writes header
`id,name,price,stock` with `newline=""` + `encoding="utf-8"`, returns the data-row count.

**Constraints**
- **`params=` carries the query; `raise_for_status()` guards the parse.**
- **CSV uses `csv.DictWriter` with exactly the fieldnames above.**
- Driver imports `import search_client as sc`.

**Coverage**
`params=` search → lesson.md Core Concept 2; DictWriter persistence → L12 retrieval.

**Expected** (Driver cell):
```
latte -> ['latte']
coffee tea -> []
muffin -> ['muffin']
wrote 3 rows
id,name,price,stock
```


In [2]:
import tempfile
from pathlib import Path

import local_api
import search_client as sc

api = local_api.LocalAPI()
try:
    for term in ["latte", "coffee tea", "muffin"]:
        data = sc.search(api.base, term)
        names = [item["name"] for item in data["items"]]
        print(term, "->", names)
    with tempfile.TemporaryDirectory() as td:
        p = Path(td) / "hits.csv"
        n = sc.save_csv(sc.search(api.base, "e")["items"], p)
        print("wrote", n, "rows")
        print(p.read_text(encoding="utf-8").splitlines()[0])
finally:
    api.stop()

latte -> ['latte']
coffee tea -> []
muffin -> ['muffin']
wrote 3 rows
id,name,price,stock


## H1 — Debug drill: fix the buggy weather client

## Intent
**Diagnose like an engineer: fix four planted `requests` bugs traceback-first, each mapping to a lesson Pitfall.**

**Type:** DEBUG — Hard — edits 1 file

**Task**
`weather_client.py` is provided **with four planted bugs**. Find and fix all
four so the suite goes green. The test file and the server are correct — you
may read them but must not modify them.

**File tree**

```
weather_client.py     (provided, buggy — YOU fix it)
test_weather.py       (provided — do NOT modify)
local_api.py          (provided — do NOT modify)
```

**Hints**
- Bug 1: `tokyo` should decode, but unknown cities vanish into `{}` instead of
  raising with a status. A 404 is a verdict, not an empty answer — which call
  turns a status into an exception?
- Bug 2: every city 404s although the cities exist. On a GET the query belongs
  in the URL — which keyword puts it there, and which one silently drops it
  into a body nobody reads?
- Bug 3: `atlantis` must raise with `status == 404`, but arrives as a generic
  network error. Which except clause catches it first?
- Bug 4: the payload decodes fine but the lookup fails. Compare the keys you
  read against `WEATHER` in `local_api.py`.

**Bug report** (fill in as you go — see the next cell).

**Coverage**
`params=` vs `data=` + `raise_for_status()` + handler order + exact keys →
lesson.md Pitfalls 1–3 + Core Concepts 2/5/6; protocol → `notes/DEBUGGING.md`.

**Expected** (Driver cell): `4 passed` — the final state after all four fixes.


### H1 bug report — blank write-in (fill each row as you fix it, never after)

```
# | Expected vs Observed                         | Root cause | Fix
--|----------------------------------------------|------------|----
1 |                                              |            |
2 |                                              |            |
3 |                                              |            |
4 |                                              |            |
```


1. bug 1: using data instead of params, change it to params
2. bug 2: if else for the code, use resp.raise_for_status() instead
3. key 'temp', change to 'temp_c'
4. the exception order is wrong, it should be catch requests.httperror first

In [5]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_weather.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

4 passed


## H2 — A Session-based client with headers-once

## Intent
**Stop repeating yourself: carry the API key and settings in one `Session` every call reuses.**

**Type:** REAL-WORLD — Hard — writes 1 file

**Task**
Pasting the auth header into every function is duplication waiting to leak.
Build `sess_client.py`: a small module where `make_session` attaches the key
once and `fetch_with` runs every request through that session.

**File tree**

```
sess_client.py        (you create)
test_sess.py          (provided — do NOT modify)
```

**Interface contract**
`ShopError(message, status=None)` with `.message` / `.status`.
`make_session(api_key: str) -> requests.Session` — the session's
`Authorization` header is exactly `f"Bearer {api_key}"`.
`fetch_with(session: requests.Session, base: str, path: str) -> dict` —
`session.get(base + path, timeout=3)` + `raise_for_status()` + `.json()`;
`HTTPError` → `ShopError` with the status; `ConnectionError`/`Timeout` →
`ShopError` with `status=None`.

**Constraints**
- **The key is set once via `sess.headers.update(...)` — never passed per call.**
- **Handlers stay in order: `HTTPError`, then `ConnectionError`, then `Timeout`.**
- Close sessions you open in the tests' `finally` (already written for you).

**Coverage**
`Session` + headers-once → lesson.md Core Concept 7; typed-error wrap → Core Concept 6.

**Expected** (Driver cell): `3 passed`.


In [6]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_sess.py", "-q", "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr)
_m = re.search(r"(\d+ passed)", _out)
print(_m.group(1) if _m else _out[-500:])

3 passed


## Challenge — A CLI that syncs API data into a local cache

## Intent
**Ship the real pattern: fetch the network set, merge it over the local cache with network-wins semantics, and persist JSON + CSV.**

**Type:** REAL-WORLD — Challenge — writes 1 file

**Task**
Real programs rarely hit the API for every screen. Build `sync_cli.py`, a
one-word CLI that fetches `/items` through its own server, merges with an
existing local cache (offline-only items survive), and writes both `sync.json`
and `sync.csv`.

**File tree**

```
sync_cli.py           (you create)
```

**Interface contract**
`python sync_cli.py sync`:
1. starts its own `local_api.LocalAPI()`,
2. fetches `/items` with `requests` (`timeout=`, `raise_for_status()`),
3. reads `sync.json` from the current directory if it exists (the cached set),
4. merges via `merge(network, cached) -> (items, fresh, kept)`: **every network id wins; cached-only items are kept; output sorted by id**,
5. writes `sync.json` (`{"items": [...]}`) and `sync.csv` (header
   `id,name,price,stock`),
6. prints exactly `Sync: N items (fresh X, cached Y)` where `fresh = len(network)`.

Any other argv (missing or unknown word) → `usage: python sync_cli.py sync` on
**stderr** with exit **1**, writing no files.

**Constraints**
- **Extract `merge(...)` so the logic is testable without the network.**
- **`requests` does the fetch; `pathlib` + `csv.DictWriter` do the persistence.**
- **One positional word only (`sync`) — no `--flags` in this lesson.**
- Test 1: seed the cache with offline-only `{"id": 5, "name": "donut", "price": 1.0, "stock": 3}` → merged ids `[1, 2, 3, 4, 5]`, summary `Sync: 5 items (fresh 4, cached 1)`.
- Test 2: no existing cache → `Sync: 4 items (fresh 4, cached 0)`.
- Test 3: bare `python sync_cli.py` → exit 1 with the usage line on stderr.
- The driver runs the CLI with `subprocess.run`, `cwd=tmp_path`, and
  `PYTHONPATH=<lesson folder>` so it imports `local_api` while writing files
  in the temp dir.

**Coverage**
fetch + merge + dual persistence → lesson.md Core Concepts 3/5 + L9/L11/L12 retrieval; positional argv validation → Stage-1 retrieval.

**Expected** (Driver cell):
```
Sync: 5 items (fresh 4, cached 1)
ids: [1, 2, 3, 4, 5]
id,name,price,stock
Sync: 4 items (fresh 4, cached 0)
usage exit: 1
```


In [7]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

lesson = Path.cwd()
with tempfile.TemporaryDirectory() as td:
    tmp = Path(td)
    (tmp / "sync.json").write_text(
        json.dumps({"items": [{"id": 5, "name": "donut", "price": 1.0, "stock": 3}]}),
        encoding="utf-8",
    )
    env = {**os.environ, "PYTHONPATH": str(lesson)}
    r1 = subprocess.run(
        [sys.executable, str(lesson / "sync_cli.py"), "sync"],
        cwd=tmp, capture_output=True, text=True, env=env,
    )
    print(r1.stdout.strip())
    merged = json.loads((tmp / "sync.json").read_text(encoding="utf-8"))
    print("ids:", [it["id"] for it in merged["items"]])
    print((tmp / "sync.csv").read_text(encoding="utf-8").splitlines()[0])
with tempfile.TemporaryDirectory() as td:
    tmp = Path(td)
    env = {**os.environ, "PYTHONPATH": str(lesson)}
    r2 = subprocess.run(
        [sys.executable, str(lesson / "sync_cli.py"), "sync"],
        cwd=tmp, capture_output=True, text=True, env=env,
    )
    print(r2.stdout.strip())
with tempfile.TemporaryDirectory() as td:
    env = {**os.environ, "PYTHONPATH": str(lesson)}
    r3 = subprocess.run(
        [sys.executable, str(lesson / "sync_cli.py")],
        cwd=td, capture_output=True, text=True, env=env,
    )
    print("usage exit:", r3.returncode)

Sync: 5 items (fresh 4, cached 1)
ids: [1, 2, 3, 4, 5]
id,name,price,stock
Sync: 4 items (fresh 4, cached 0)
usage exit: 1
